# FORWARD CHAINING

**Data-driven** _(a.k.a. *bottom-up*, *forward*)_ production-rule engine: On every pass the engine scans the whole rule base and fires *every* rule whose premises are already satisfied, instead of stopping at the first match. It keeps looping until a full pass adds no new fact _(a *fixpoint*)_, which is the standard termination condition for forward chaining over a finite, monotonic rule base _(facts are only ever added, never retracted)_.

---

In [1]:
versioninfo()  # -> v"1.11.7"

Julia Version 1.11.7
Commit f2b3dbda30a (2025-09-08 12:10 UTC)
Build Info:
  Official https://julialang.org/ release
Platform Info:
  OS: Linux (x86_64-linux-gnu)
  CPU: 8 × Intel(R) Core(TM) i7-8565U CPU @ 1.80GHz
  WORD_SIZE: 64
  LLVM: libLLVM-16.0.6 (ORCJIT, skylake)
Threads: 1 default, 0 interactive, 1 GC (on 8 virtual cores)
Environment:
  JULIA_GPG = 3673DF529D9049477F76B37566E3C7DC03D6E495
  JULIA_PATH = /usr/local/julia
  JULIA_DEPOT_PATH = /root/.julia
  JULIA_VERSION = 1.11.7


---

## Data Structure

**Representation.** Facts and rule premises/conclusions are represented as `Symbol`s _(e.g. `:has_hair`, `:is_mammal`)_. For a single individual, a first-order predicate like `has_hair(a)` is *propositionalized* into one ground symbol `:has_hair`. We don't track *which* individual `a` is, because there's only ever one animal under consideration at a time.

In [2]:
module Chaining

    export ¬, Rule
    export forward_chain!
    
    include("Chaining.jl")

end

Main.Chaining

In [3]:
using .Chaining

A production rule: IF all `premises` are in working memory THEN `conclusion` can be added to working memory.

## The Rule Base

In [4]:
# A vector of `Rule`s representing the rule base.
const RULES = Rule[
    Rule("R1",  [:has_hair], :is_mammal),
    Rule("R2",  [:gives_milk], :is_mammal),
    Rule("R3",  [:has_feathers], :is_bird),
    Rule("R4",  [:flies, :lays_eggs], :is_bird),
    Rule("R5",  [:is_mammal, :eats_meat], :is_carnivore),
    Rule("R6",  [:is_mammal, :has_pointed_teeth, :has_claws, :has_forward_eyes], :is_carnivore),
    Rule("R7",  [:is_carnivore, :has_tawny_color, :has_dark_spots], :is_cheetah),
    Rule("R8",  [:is_carnivore, :has_tawny_color, :has_black_stripes], :is_tiger),
    Rule("R9",  [:is_bird, ¬(:flies), :has_long_neck, :has_long_legs], :is_ostrich),
    Rule("R10", [:is_bird, ¬(:flies), :swims, :has_black_white_color], :is_penguin),
    Rule("R11", [:is_bird, :is_good_flyer], :is_albatross)
]

11-element Vector{Rule}:
 Rule("R1", Union{Main.Chaining.Neg, Symbol}[:has_hair], :is_mammal)
 Rule("R2", Union{Main.Chaining.Neg, Symbol}[:gives_milk], :is_mammal)
 Rule("R3", Union{Main.Chaining.Neg, Symbol}[:has_feathers], :is_bird)
 Rule("R4", Union{Main.Chaining.Neg, Symbol}[:flies, :lays_eggs], :is_bird)
 Rule("R5", Union{Main.Chaining.Neg, Symbol}[:is_mammal, :eats_meat], :is_carnivore)
 Rule("R6", Union{Main.Chaining.Neg, Symbol}[:is_mammal, :has_pointed_teeth, :has_claws, :has_forward_eyes], :is_carnivore)
 Rule("R7", Union{Main.Chaining.Neg, Symbol}[:is_carnivore, :has_tawny_color, :has_dark_spots], :is_cheetah)
 Rule("R8", Union{Main.Chaining.Neg, Symbol}[:is_carnivore, :has_tawny_color, :has_black_stripes], :is_tiger)
 Rule("R9", Union{Main.Chaining.Neg, Symbol}[:is_bird, Main.Chaining.Neg(:flies), :has_long_neck, :has_long_legs], :is_ostrich)
 Rule("R10", Union{Main.Chaining.Neg, Symbol}[:is_bird, Main.Chaining.Neg(:flies), :swims, :has_black_white_color], :is_penguin)
 Ru

## Demo Using a Concrete Animal

The `initial_facts` below (hair, tawny color, dark spots, pointed teeth, claws, forward-facing eyes) are chosen so that:

- $\mathfrak{R}_1$ fires immediately (`has_hair` → `is_mammal`),
- which then unblocks $\mathfrak{R}_6$ (`is_mammal` + teeth/claws/eyes → `is_carnivore`),
- which then unblocks $\mathfrak{R}_7$ (`is_carnivore` + tawny + dark spots → `is_cheetah`).

So we expect the engine to converge on **CHEETAH** after a few passes, deriving `is_mammal`, `is_carnivore`, and `is_cheetah` along the way. 

Note `eats_meat` is *not* in the initial facts, so $\mathfrak{R}_5$ never fires. $\mathfrak{R}_6$ is the only path to `is_carnivore` here, which is a nice illustration that the engine doesn't care which of several valid paths gets used.

In [5]:
function main()
    
    initial_facts = Set{Symbol}([
        :has_hair,
        :has_tawny_color,
        :has_dark_spots,
        :has_pointed_teeth,
        :has_claws,
        :has_forward_eyes,
    ])

    animals = Dict([
            (:is_albatross, "ALBATROSS"),
            (:is_bird, "BIRD"),
            (:is_cheetah, "CHEETAH"),
            (:is_penguin, "PENGUIN"),
            (:is_tiger, "TIGER")
            ])

    println("Initial facts: ", initial_facts)

    #= Whenever a rule fires, we `push!` its conclusion into `wm` and set `changed = true`,  which forces another full pass.
    A newly derived fact _(e.g., `:is_mammal`)_ might unblock a rule that couldn't fire earlier _(e.g., $\mathfrak{R}_6$), 
    which needs `:is_mammal` as one of its premises)_. 
    
    The loop stops the first time a whole pass fires nothing _(i.e., at the fixpoint)_. =#
    
    wm = forward_chain!(copy(initial_facts), RULES)  # copy() so the caller's initial_facts set is left untouched

    println("\nFinal working memory: ", wm)
    derived = setdiff(wm, initial_facts)
    println("\nFacts derived by the engine: ", derived)

    for animal in animals
        k, v = animal.first, animal.second
        if k in wm
            println("\nConclusion: the animal is a $v.")
            break
        else
            println("\nNo specific species could be concluded from the given facts.")   
        end
    end
    
end

main (generic function with 1 method)

In [6]:
main()

Initial facts: Set([:has_claws, :has_forward_eyes, :has_tawny_color, :has_pointed_teeth, :has_hair, :has_dark_spots])

--- PASS 1 ---
  Fired R1: has_hair -> is_mammal   [new fact: is_mammal]
  Fired R6: is_mammal ∧ has_pointed_teeth ∧ has_claws ∧ has_forward_eyes -> is_carnivore   [new fact: is_carnivore]
  Fired R7: is_carnivore ∧ has_tawny_color ∧ has_dark_spots -> is_cheetah   [new fact: is_cheetah]

--- PASS 2 ---
--- Fixpoint reached: no more rules apply ---

Final working memory: Set([:is_carnivore, :is_cheetah, :has_claws, :has_forward_eyes, :has_tawny_color, :has_pointed_teeth, :is_mammal, :has_hair, :has_dark_spots])

Facts derived by the engine: Set([:is_carnivore, :is_cheetah, :is_mammal])

Conclusion: the animal is a CHEETAH.
